# 🏰 Your Data Is Your Moat

Generic models are **commodities**. Everyone has access to the same GPT-4o.

**YOUR data**, combined with systematic tuning, creates something competitors can't replicate.

This notebook uses the project's real CSV ticket data to prove it.

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, pandas as pd, ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.actions import run_baseline, run_optimization, compare_models
from dspy_tasks.visualize import *

# Show the real ticket data
df = pd.read_csv("../csv/data.csv", encoding="latin-1")
print(f"📊 Real ticket data: {len(df)} tickets")
print(f"Columns: {', '.join(df.columns[:10])}...")
df[["Summary*", "Priority*", "Status*", "Assigned Group*+"]].head(5)

In [ ]:
task = get_task("ticket_routing")
examples = task.load_examples()
print(f"Training data: {len(examples)} ticket examples\n")
for ex in examples[:3]:
    print(f"  Summary: {str(ex.summary)[:80]}...")
    print(f"  → Category: {ex.category} | Priority: {ex.priority} | Team: {ex.assigned_group}\n")

## Generic Prompt vs. Domain-Tuned

A generic prompt says: *"Route this ticket to the right team."*

A domain-tuned prompt knows YOUR team names, YOUR categories, YOUR routing patterns —
learned automatically from YOUR ticket history.

In [ ]:
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy
MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
btn = run_button("Generic vs. Tuned")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        print(f"⏳ Running ticket routing: generic vs. domain-tuned on {model_dd.value}...")

        result = run_optimization("ticket_routing", model_dd.value, "BootstrapFewShot", max_eval=10)

        display_improvement(result.baseline_score, result.optimized_score)
        display_prompt_diff(result.prompt_before, result.prompt_after,
            title="Generic Prompt vs. Domain-Tuned Prompt")

        display_insight("The Moat",
            f"A generic prompt scores {result.baseline_score:.0%}. "
            f"Tuned with YOUR ticket data: {result.optimized_score:.0%}. "
            "This domain knowledge is YOUR competitive advantage — "
            "no competitor can replicate your data.")

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
compare_btn = run_button("Compare Models on Domain Tasks")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        result = compare_models("ticket_routing", MODELS, max_eval=8)

        model_scores = {}
        for m in MODELS:
            short = m.split("/")[-1]
            model_scores[short] = {
                "baseline": result.baseline_scores[m],
                "optimized": result.optimized_scores[m],
            }
        fig = bar_comparison("Ticket Routing: Model Comparison", model_scores)
        fig.show()

        # Check if small model + tuning beats big model
        if len(MODELS) >= 2:
            small_opt = result.optimized_scores.get(MODELS[1], 0)
            big_base = result.baseline_scores.get(MODELS[0], 0)
            if small_opt > big_base:
                display_insight("💰 The Cost Insight",
                    f"{MODELS[1].split('/')[-1]} optimized ({small_opt:.0%}) beats "
                    f"{MODELS[0].split('/')[-1]} unoptimized ({big_base:.0%})! "
                    "A tuned small model beats an untuned large model — and costs 10x less.")

compare_btn.on_click(on_compare)
display(compare_btn, compare_out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task("comparative_analysis"), get_task("instruction_constraints")]],
    description="Task:")
run_btn = run_button("Run & Optimize")
run_out = widgets.Output()

def on_run_extra(b):
    with run_out:
        run_out.clear_output()
        result = run_optimization(task_dd.value, model_dd.value, max_eval=8)
        display_improvement(result.baseline_score, result.optimized_score)
        display_results_table(result.individual_scores if hasattr(result, 'individual_scores') else [])

run_btn.on_click(on_run_extra)
display(widgets.HBox([task_dd, run_btn]), run_out)

## Key Takeaway

**Your data + systematic tuning = your moat.**

- The **model** is rented (API access anyone can buy)
- The **data** is owned (your ticket history, your domain)
- The **tuning** is your engineering (DSPy compilation with your metrics)

Together, they create a system that gets better as you collect more data —
a flywheel your competitors can't copy.